In [ ]:
import os
import json
import shap
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

model = joblib.load('models/anovator_age_gap_model.joblib')
scaler = joblib.load('models/anovator_biological_scaler.joblib')

with open('models/model_features.json', 'r') as f:
    required_features = json.load(f)

# Load training-set medians for safe imputation (generated by Health_Pipeline.ipynb)
try:
    with open('models/training_medians.json', 'r') as f:
        training_medians = json.load(f)
    print(f"Training medians loaded: {len(training_medians)} feature medians available.")
except FileNotFoundError:
    training_medians = {}
    print("WARNING: models/training_medians.json not found. Re-run Health_Pipeline.ipynb.")

df_pop = pd.read_csv('data/Anovator_Biological_Master.csv')

In [ ]:
def preprocess_client_scan(df_input, features_list):
    """Transforms a raw scan row into a model-ready feature matrix."""
    df_proc = df_input.copy()

    if 'sex' in df_proc.columns:
        df_proc['sex_encoded'] = df_proc['sex'].map({'F': 0, 'M': 1})

    df_proc['muscle_fat_ratio'] = df_proc['muscle'] / (df_proc['fat'] + 0.1)
    df_proc['upper_lower_muscle_ratio'] = df_proc['upperBody'] / (df_proc['lowerBody'] + 0.1)

    limb_fat_cols = ['fatLeftArm', 'fatRightArm', 'fatLeftLeg', 'fatRightLeg']
    if all(col in df_proc.columns for col in limb_fat_cols):
        df_proc['trunk_limb_fat_ratio'] = df_proc['fatTrunk'] / (df_proc[limb_fat_cols].sum(axis=1) + 0.1)
    else:
        df_proc['trunk_limb_fat_ratio'] = 0.5

    postural_risk_cols = ['humpbackRisk', 'spineRisk', 'pelvisRisk', 'postureRisk', 'kneeRisk', 'frontHeadRisk']
    df_proc['aggregated_postural_index'] = df_proc[postural_risk_cols].mean(axis=1)

    df_final = df_proc.reindex(columns=features_list)

    # Step 1: fill with training-set medians (the "average patient" imputation)
    df_final = df_final.fillna(training_medians)

    # Step 2: override specific vital signs with established clinical averages
    clinical_defaults = {'restingHeartRate': 70.0, 'bloodMaxPressure': 120.0, 'bloodMinPressure': 80.0}
    for col, val in clinical_defaults.items():
        if col in df_final.columns:
            df_final[col] = df_final[col].fillna(val)

    # Step 3: last-resort fallback
    df_final = df_final.fillna(0)
    return df_final

print("Preprocessing engine ready (uses training-set medians for imputation).")

In [ ]:
def simulate_health_intervention(baseline_scan, fat_reduction=0, muscle_gain=0, wc_improvement=0):
    """
    Predicts the biological age gap after hypothetical body composition improvements.
    Fat loss is applied proportionally across all fat segments so derived ratios
    (trunk_limb_fat_ratio) stay internally consistent.  BMI is recomputed from the
    updated weight so the model does not see a contradictory weight/BMI pair.
    """
    sim_data = baseline_scan.copy()

    fat_original = float(sim_data['fat'].iloc[0])
    # Clip to a biologically plausible minimum (1 kg total body fat)
    fat_new = max(1.0, fat_original - fat_reduction)
    actual_fat_red = fat_original - fat_new

    # Scale every fat segment by the same ratio so the trunk-to-limb ratio is preserved
    if actual_fat_red > 0 and fat_original > 0:
        fat_scale = fat_new / fat_original
        for seg in ['fatLeftArm', 'fatRightArm', 'fatLeftLeg', 'fatRightLeg', 'fatTrunk']:
            if seg in sim_data.columns:
                sim_data[seg] = sim_data[seg] * fat_scale

    sim_data['fat']    = fat_new
    sim_data['weight'] = sim_data['weight'] - actual_fat_red
    # Recompute BMI so weight and BMI are consistent (height is unchanged)
    sim_data['bmi']    = sim_data['weight'] / (sim_data['height'] / 100) ** 2
    sim_data['muscle'] = sim_data['muscle'] + muscle_gain
    sim_data['wc']     = sim_data['wc'] - wc_improvement

    df_prepared = preprocess_client_scan(sim_data, required_features)
    X_scaled    = scaler.transform(df_prepared)
    return model.predict(X_scaled)[0]

In [17]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# Creating an output widget to display the dynamic health reports
report_output = widgets.Output()

def update_dashboard(change):
    """
    Recalculating the health risk report every time a slider is moved.
    Researching the real-time impact of body composition changes on longevity.
    """
    # Selecting a specific client scan to act as the baseline for the simulation
    # Using index 10 from the population as a research sample
    client_scan = df_pop.iloc[[10]]
    
    # Calculating the original biological age gap before any intervention
    df_orig = preprocess_client_scan(client_scan, required_features)
    X_orig_scaled = scaler.transform(df_orig)
    original_gap = model.predict(X_orig_scaled)[0]
    
    # Executing the simulation using the current values from the interactive sliders
    new_predicted_gap = simulate_health_intervention(
        client_scan, 
        fat_reduction=fat_slider.value, 
        muscle_gain=muscle_slider.value, 
        wc_improvement=wc_slider.value
    )
    
    # Refreshing the report display area with the updated simulation results
    with report_output:
        clear_output(wait=True)
        print("="*60)
        print(f" ZDRAVOLETIE INTERACTIVE LONGEVITY SIMULATOR")
        print("="*60)
        print(f"Baseline Age Gap:  {original_gap:.2f} years")
        print(f"Simulated Age Gap: {new_predicted_gap:.2f} years")
        print("-" * 60)
        
        # Calculating the 'Biological Gain' to show the effectiveness of the intervention
        years_saved = original_gap - new_predicted_gap
        if years_saved > 0:
            print(f"SUCCESS: These changes result in a {years_saved:.2f} year reduction in biological age.")
        else:
            print("Current adjustments have a neutral impact on biological aging.")
        print("-" * 60)

# Creating interactive sliders for body fat, muscle mass, and waist-to-hip ratio
fat_slider = widgets.FloatSlider(value=0, min=0, max=5.0, step=0.1, description='Fat Loss (kg)')
muscle_slider = widgets.FloatSlider(value=0, min=0, max=3.0, step=0.1, description='Muscle Gain (kg)')
wc_slider = widgets.FloatSlider(value=0, min=0, max=0.05, step=0.005, description='WC Improve')

# Attaching the update logic to trigger on any slider value change
fat_slider.observe(update_dashboard, names='value')
muscle_slider.observe(update_dashboard, names='value')
wc_slider.observe(update_dashboard, names='value')

# Grouping the sliders and report output into a professional dashboard layout
dashboard_layout = widgets.VBox([
    widgets.HTML("<b>Adjust parameters to simulate a health intervention:</b>"),
    fat_slider, 
    muscle_slider, 
    wc_slider, 
    report_output
])

# Displaying the finalized interactive simulation tool
display(dashboard_layout)

# Initializing the first report calculation
update_dashboard(None)

In [ ]:
%%writefile zdravoletie_app.py
import streamlit as st
import pandas as pd
import numpy as np
import joblib
import json
import shap
import matplotlib.pyplot as plt

st.set_page_config(page_title="Zdravoletie AI", layout="wide")

@st.cache_resource
def load_assets():
    model = joblib.load('models/anovator_age_gap_model.joblib')
    scaler = joblib.load('models/anovator_biological_scaler.joblib')
    with open('models/model_features.json', 'r') as f:
        features = json.load(f)
    # Load training-set medians for safe imputation at inference time.
    # These prevent the model from receiving clinically impossible values like 0 vital capacity.
    try:
        with open('models/training_medians.json', 'r') as f:
            medians = json.load(f)
    except FileNotFoundError:
        medians = {}
        st.warning("models/training_medians.json not found — re-run Health_Pipeline.ipynb to generate it.")
    return model, scaler, features, medians

model, scaler, required_features, training_medians = load_assets()

# Cache the SHAP explainer — building it once per session avoids rebuilding on every slider move
@st.cache_resource
def get_explainer(_model):
    return shap.TreeExplainer(_model)

explainer = get_explainer(model)


def preprocess_input(df_input, features_list):
    """Transforms a raw scan row into a scaled model-ready matrix."""
    df_proc = df_input.copy()
    if 'sex' in df_proc.columns:
        df_proc['sex_encoded'] = df_proc['sex'].map({'F': 0, 'M': 1})
    df_proc['muscle_fat_ratio'] = df_proc['muscle'] / (df_proc['fat'] + 0.1)
    df_proc['upper_lower_muscle_ratio'] = df_proc['upperBody'] / (df_proc['lowerBody'] + 0.1)
    limb_fat = ['fatLeftArm', 'fatRightArm', 'fatLeftLeg', 'fatRightLeg']
    df_proc['trunk_limb_fat_ratio'] = df_proc['fatTrunk'] / (df_proc[limb_fat].sum(axis=1) + 0.1)
    postural = ['humpbackRisk', 'spineRisk', 'pelvisRisk', 'postureRisk', 'kneeRisk', 'frontHeadRisk']
    df_proc['aggregated_postural_index'] = df_proc[postural].mean(axis=1)
    df_final = df_proc.reindex(columns=features_list)
    # Fill with training-set medians first; only fall back to 0 for anything not covered
    df_final = df_final.fillna(training_medians).fillna(0)
    return df_final


# ── UI ──────────────────────────────────────────────────────────────────────
st.title("🧬 Zdravoletie Biometric Intelligence Dashboard")
st.markdown("---")

st.sidebar.header("Client Data Selection")
df_pop = pd.read_csv("data/Anovator_Biological_Master.csv")
client_idx = st.sidebar.number_input("Select Client Index", min_value=0, max_value=len(df_pop)-1, value=10)
client_scan = df_pop.iloc[[client_idx]]

df_base = preprocess_input(client_scan, required_features)
baseline_gap = model.predict(scaler.transform(df_base))[0]

st.sidebar.header("Simulate Health Intervention")
fat_red  = st.sidebar.slider("Target Fat Loss (kg)", 0.0, 10.0, 0.0, step=0.1)
mus_gain = st.sidebar.slider("Target Muscle Gain (kg)", 0.0, 5.0, 0.0, step=0.1)
wc_imp   = st.sidebar.slider("WC Improvement", 0.0, 0.10, 0.0, step=0.005, format="%.3f")

# ── Simulation ──────────────────────────────────────────────────────────────
sim_scan = client_scan.copy()

fat_original = float(sim_scan['fat'].iloc[0])
# Clip to a biologically plausible minimum (1 kg total fat)
fat_new = max(1.0, fat_original - fat_red)
actual_fat_red = fat_original - fat_new

# Scale all fat segments proportionally so derived ratios (trunk_limb_fat_ratio) stay consistent
if actual_fat_red > 0 and fat_original > 0:
    fat_scale = fat_new / fat_original
    for seg in ['fatLeftArm', 'fatRightArm', 'fatLeftLeg', 'fatRightLeg', 'fatTrunk']:
        if seg in sim_scan.columns:
            sim_scan[seg] = sim_scan[seg] * fat_scale

sim_scan['fat']    = fat_new
sim_scan['weight'] = sim_scan['weight'] - actual_fat_red
# Recompute BMI from the updated weight — weight changed but height did not
sim_scan['bmi']    = sim_scan['weight'] / (sim_scan['height'] / 100) ** 2
sim_scan['muscle'] = sim_scan['muscle'] + mus_gain
sim_scan['wc']     = sim_scan['wc'] - wc_imp

df_sim       = preprocess_input(sim_scan, required_features)
X_sim_scaled = scaler.transform(df_sim)
simulated_gap = model.predict(X_sim_scaled)[0]

# ── Metrics ─────────────────────────────────────────────────────────────────
col1, col2, col3 = st.columns(3)
col1.metric("Baseline Age Gap",  f"{baseline_gap:.2f} y")
col2.metric("Simulated Age Gap", f"{simulated_gap:.2f} y",
            delta=f"{simulated_gap - baseline_gap:.2f}", delta_color="inverse")
col3.metric("Longevity Gain",    f"{baseline_gap - simulated_gap:.2f} years")

# ── SHAP force plot ──────────────────────────────────────────────────────────
st.subheader("Individual Biometric Driver Analysis")
shap_values = explainer.shap_values(X_sim_scaled)

fig = plt.figure(figsize=(12, 3))
shap.force_plot(
    explainer.expected_value,
    shap_values[0, :],
    np.round(X_sim_scaled[0, :], 2),
    feature_names=required_features,
    matplotlib=True,
    show=False
)
st.pyplot(plt.gcf())
plt.close()

In [19]:
# Executing the dashboard script to launch the local web server
# Researching the deployment of clinical tools for real-time stakeholder interaction

!streamlit run zdravoletie_app.py

^C
